# 02 - Ingeniería de Características y Segmentación por Zonas de Interés (ZOI)
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. Fundamento Teórico de la Ingeniería de Características
Para superar la subjetividad de etiquetas como *agresivo* o *suave*, descomponemos cada curva del circuito en tres **Zonas de Interés (ZOI)**:
1. **Fase de Frenado (*Braking Zone*)**: Desde la iniciación de la presión de freno hasta el vértice.
   - *Métricas*: Distancia de inicio de frenado al vértice ($m$), Máxima desaceleración longitudinal ($m/s^2$).
2. **Fase de Vértice (*Apex Zone*)**: Ventana espacial inmediata alrededor del punto de mínima distancia angular.
   - *Métricas*: Velocidad mínima en vértice ($km/h$), Nivel de *Trail Braking* (frenado en rotación).
3. **Fase de Salida (*Exit Zone*)**: Región de reaplicación del acelerador y tracción.
   - *Métricas*: Tasa máxima de aplicación del acelerador ($\%/s$), Distancia al inicio de potencia positiva ($m$).

Posteriormente, aplicamos **Estandarización Z-score** para nivelar las magnitudes físicas y permitir la comparabilidad multidimensional.


In [ ]:
import fastf1 as ff1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_style_clustering.data.loader import load_session, enable_cache
from f1_style_clustering.features.zoi import extract_circuit_zones, get_corner_fragments
from f1_style_clustering.features.aggregators import build_session_feature_matrix
from f1_style_clustering.features.preprocessor import normalize_relative_to_mean, standardize_features
from f1_style_clustering.visualization.telemetry_plots import plot_corner_telemetry_comparison

enable_cache('cache')
session = load_session(year=2025, grand_prix='Italian Grand Prix', session_type='R')
print('Sesión cargada:', session.event.EventName)


## 2. Extracción Dinámica de Zonas de Interés (ZOI)

In [ ]:
# Extraer ZOIs para todas las curvas del circuito
analysis_zones = extract_circuit_zones(session)
print(f'Total de curvas extraídas: {len(analysis_zones)}')
for c_name, z_info in list(analysis_zones.items())[:3]:
    print(f"{c_name}: Frenado {z_info['braking']}, Vértice {z_info['apex']}, Salida {z_info['exit']}")


## 3. Visualización de Perfiles Telemétricos en Curvas Clave

In [ ]:
# Curva 1 (Variante del Rettifilo)
t1_zone = analysis_zones['Turn_1']
fig_t1 = plot_corner_telemetry_comparison(session, t1_zone, corner_title='Turn 1 - Rettifilo')
plt.show()


## 4. Construcción de la Matriz de Características de Pilotos

In [ ]:
# Construir matriz de features agregada para toda la sesión
df_raw = build_session_feature_matrix(session, analysis_zones=analysis_zones)
print(f'Matriz de features crudas: {df_raw.shape[0]} pilotos x {df_raw.shape[1]} columnas')
display(df_raw.head())


## 5. Normalización y Estandarización a Z-Scores

In [ ]:
# 1. Desviación relativa respecto a la media de la sesión
df_norm, df_meta = normalize_relative_to_mean(df_raw)

# 2. Estandarización Z-score (Media = 0, Desv. Est. = 1)
df_standardized, _, scaler = standardize_features(df_norm)

print('--- Matriz Estandarizada (Z-Scores) lista para Reducción y Clustering ---')
display(df_standardized.head())


In [ ]:
# Mapa de calor de características estandarizadas
plt.figure(figsize=(14, 7))
sns.heatmap(df_standardized.iloc[:, :12], cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.5)
plt.title('Mapa de Calor de Z-Scores de Estilo de Conducción (Pilotos vs Métricas de Curva)', fontsize=12, fontweight='bold')
plt.xlabel('Métricas de Telemetría (Z-score)')
plt.ylabel('Piloto')
plt.tight_layout()
plt.show()
